[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap05_visualizacao_estatistica/exemplo_visualizacao_prazos_entrega.ipynb)

**Capítulo 5 | Visualização estatística**  
**Base:** Caso Tech, planilha `logística`  
**Unidade de análise:** pedido com registro logístico

## Objetivo

Explorar a distribuição dos prazos de entrega e comparar os resultados entre transportadoras por meio de histogramas, boxplots e gráficos de comparação entre grupos. Cada visualização está associada a uma pergunta analítica. A análise usa a tabela logística já estruturada no arquivo do GitHub; não agrega a tabela `Vendas`, pois as variáveis necessárias para este exemplo estão na própria tabela `logística`.

## Perguntas orientadoras

1. Como se distribuem os dias entre a entrega prevista e a entrega real?
2. Qual é a concentração dos prazos e há observações atípicas?
3. As distribuições e medidas de centro diferem entre as transportadoras?
4. Qual proporção de entregas ocorreu antes, no prazo ou depois da data prevista?

## Definição operacional

A coluna `Atraso em dias` é utilizada como diferença entre a data de entrega real e a data prevista. Valores negativos indicam entrega antecipada, zero indica entrega na data prevista e valores positivos indicam atraso. Os 83 pedidos sem data de entrega real são registros sem desfecho observado e não entram nas distribuições de prazo. O campo de atraso desses casos não deve ser interpretado como duração efetiva.

> O notebook consulta a base diretamente no repositório GitHub. Para abrir no Colab a partir do repositório, salve este arquivo no caminho indicado no endereço do botão. Se o notebook for colocado em outro caminho ou branch, ajuste o link do badge.


## 1. Preparação do ambiente

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")


## 2. Leitura da base e inspeção inicial

**Pergunta:** quais registros e variáveis estão disponíveis para descrever os prazos de entrega?

In [ ]:
url_base = "https://raw.githubusercontent.com/WesleyInfoBr2/cienciadedados/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"
logistica = pd.read_excel(url_base, sheet_name="logística")

print(f"Dimensão da tabela: {logistica.shape[0]:,} linhas e {logistica.shape[1]} colunas")
display(logistica.head())
display(logistica.dtypes.to_frame("tipo de dado"))


## 3. Tratamento e definição da população analisada

In [ ]:
colunas_necessarias = [
    "Cod.Pedido",
    "Data de Entrega Prevista",
    "Data de Entrega Real",
    "Status da Entrega",
    "Transportadora",
    "Atraso em dias",
]

faltantes = [coluna for coluna in colunas_necessarias if coluna not in logistica.columns]
if faltantes:
    raise KeyError(f"Colunas esperadas não encontradas: {faltantes}")

logistica["Data de Entrega Prevista"] = pd.to_datetime(logistica["Data de Entrega Prevista"], errors="coerce")
logistica["Data de Entrega Real"] = pd.to_datetime(logistica["Data de Entrega Real"], errors="coerce")
logistica["Atraso em dias"] = pd.to_numeric(logistica["Atraso em dias"], errors="coerce")

if logistica["Cod.Pedido"].duplicated().any():
    duplicados = logistica.loc[logistica["Cod.Pedido"].duplicated(False), "Cod.Pedido"].nunique()
    raise ValueError(f"Foram encontrados {duplicados} pedidos duplicados na tabela logística.")

entregas = logistica.loc[
    logistica["Data de Entrega Real"].notna()
    & logistica["Atraso em dias"].notna()
    & logistica["Transportadora"].notna()
].copy()

entregas["Prazo recalculado (dias)"] = (
    entregas["Data de Entrega Real"] - entregas["Data de Entrega Prevista"]
).dt.days
entregas["Divergência (dias)"] = entregas["Atraso em dias"] - entregas["Prazo recalculado (dias)"]

print(f"Pedidos na tabela logística: {len(logistica):,}")
print(f"Pedidos com entrega observada: {len(entregas):,}")
print(f"Pedidos sem data de entrega real: {logistica['Data de Entrega Real'].isna().sum():,}")
print(f"Pedidos com divergência entre atraso informado e datas: {(entregas['Divergência (dias)'] != 0).sum():,}")


### Verificação do cálculo

**Pergunta:** a variável `Atraso em dias` coincide com a diferença entre a data real e a data prevista nos pedidos entregues? A verificação evita tratar como prazos observados os campos calculados para pedidos sem entrega registrada.

In [ ]:
display(
    entregas.loc[entregas["Divergência (dias)"].ne(0), [
        "Cod.Pedido", "Data de Entrega Prevista", "Data de Entrega Real",
        "Atraso em dias", "Prazo recalculado (dias)", "Divergência (dias)"
    ]].head(10)
)


## 4. Estatísticas descritivas do prazo

**Pergunta:** quais são o centro, a dispersão e os extremos da distribuição observada? A tabela complementa as visualizações.

In [ ]:
resumo_prazos = entregas["Atraso em dias"].describe().to_frame().T
resumo_prazos.index = ["Atraso em dias"]
display(resumo_prazos)

print("Frequência por valor de atraso (dias):")
display(entregas["Atraso em dias"].value_counts().sort_index().to_frame("pedidos"))


## 5. Histograma do prazo de entrega

**Pergunta:** em quais valores de atraso ou antecipação se concentram os pedidos entregues?

O eixo horizontal representa dias em relação à data prevista. Valores negativos correspondem a entregas antecipadas, zero indica entrega na data prevista e valores positivos indicam atraso. Os intervalos são alinhados aos dias inteiros registrados.

In [ ]:
limite_inferior = int(entregas["Atraso em dias"].min())
limite_superior = int(entregas["Atraso em dias"].max())

plt.figure(figsize=(9, 5))
sns.histplot(
    data=entregas,
    x="Atraso em dias",
    bins=range(limite_inferior, limite_superior + 2),
    color="#2a788e",
    edgecolor="white",
)
plt.axvline(0, color="#d1495b", linestyle="--", linewidth=1.5, label="Data prevista")
plt.title("Distribuição dos prazos de entrega observados")
plt.xlabel("Diferença entre entrega real e prevista (dias)")
plt.ylabel("Número de pedidos")
plt.xticks(range(limite_inferior, limite_superior + 1))
plt.legend()
plt.tight_layout()
plt.show()


## 6. Boxplot geral

**Pergunta:** qual é a mediana, a amplitude interquartil e a presença de valores extremos nos prazos observados?

Os pontos além dos limites do boxplot são observações a examinar, não erros automaticamente.

In [ ]:
plt.figure(figsize=(9, 3.5))
sns.boxplot(data=entregas, x="Atraso em dias", color="#73a9ad", showmeans=True)
plt.axvline(0, color="#d1495b", linestyle="--", linewidth=1.5, label="Data prevista")
plt.title("Boxplot dos prazos de entrega observados")
plt.xlabel("Diferença entre entrega real e prevista (dias)")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Comparação dos prazos por transportadora

**Pergunta:** os prazos observados apresentam diferenças de distribuição entre transportadoras?

O boxplot compara medianas e dispersão. Os grupos são ordenados pela mediana. A comparação é descritiva e não controla diferenças de rota, período ou perfil dos pedidos.

In [ ]:
ordem_transportadoras = (
    entregas.groupby("Transportadora")["Atraso em dias"]
    .median()
    .sort_values()
    .index
)

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=entregas,
    x="Transportadora",
    y="Atraso em dias",
    order=ordem_transportadoras,
    color="#8fc0a9",
    showmeans=True,
)
plt.axhline(0, color="#d1495b", linestyle="--", linewidth=1.5, label="Data prevista")
plt.title("Prazos de entrega por transportadora")
plt.xlabel("Transportadora")
plt.ylabel("Diferença entre entrega real e prevista (dias)")
plt.legend()
plt.tight_layout()
plt.show()


### Resumo numérico por grupo

**Pergunta:** como se comparam número de pedidos, média, mediana e dispersão por transportadora?

In [ ]:
resumo_transportadora = (
    entregas.groupby("Transportadora")["Atraso em dias"]
    .agg(pedidos="count", media="mean", mediana="median", desvio_padrao="std", minimo="min", maximo="max")
    .sort_values("mediana")
)
display(resumo_transportadora.round(2))


## 8. Proporção de entregas antecipadas, no prazo e atrasadas

**Pergunta:** qual proporção de pedidos entregues ocorreu antes, na data prevista ou depois dela, em cada transportadora?

As categorias são definidas pelo sinal do prazo. O gráfico empilhado compara proporções dentro de cada transportadora, não o volume absoluto.

In [ ]:
def classificar_prazo(valor):
    if valor < 0:
        return "Antecipada"
    if valor == 0:
        return "No prazo"
    return "Atrasada"

entregas["Classificação do prazo"] = entregas["Atraso em dias"].map(classificar_prazo)
ordem_classes = ["Antecipada", "No prazo", "Atrasada"]
proporcoes = pd.crosstab(
    entregas["Transportadora"], entregas["Classificação do prazo"], normalize="index"
).reindex(columns=ordem_classes, fill_value=0)

ax = proporcoes.plot(
    kind="bar", stacked=True, figsize=(10, 5),
    color=["#4c956c", "#f4a259", "#d1495b"], edgecolor="white"
)
ax.set_title("Composição dos prazos por transportadora")
ax.set_xlabel("Transportadora")
ax.set_ylabel("Proporção dos pedidos entregues")
ax.set_ylim(0, 1)
ax.set_yticks([0, .2, .4, .6, .8, 1.0], labels=["0%", "20%", "40%", "60%", "80%", "100%"])
ax.legend(title="Resultado do prazo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display((proporcoes * 100).round(1).rename_axis("Transportadora"))


## 9. Síntese da leitura

- **Distribuição geral:** o histograma mostra os valores mais frequentes e sua posição em relação à data prevista.
- **Centro e dispersão:** o boxplot e as estatísticas descritivas resumem mediana, quartis e valores que merecem verificação.
- **Comparação entre transportadoras:** leia os boxplots junto da tabela de medidas e do número de pedidos.
- **Composição dos resultados:** as proporções mostram o perfil relativo de cada grupo.

### Cuidados de interpretação

1. Os pedidos sem data real não têm prazo observado e foram excluídos das distribuições.
2. Valor negativo representa antecipação, não erro.
3. As diferenças entre transportadoras são descritivas e não demonstram causalidade.
4. A análise assume um registro por pedido na tabela logística; duplicidades são verificadas.
5. A validação da base encontrou 59 divergências de um dia entre o prazo informado e o cálculo pelas datas nos pedidos entregues. O exemplo preserva a variável registrada para manter compatibilidade com a base tratada e apresenta a divergência para investigação, sem corrigir silenciosamente os valores.

## 10. Checklist final

- [ ] Conferi nomes e tipos das colunas utilizadas.
- [ ] Verifiquei se há um registro por pedido.
- [ ] Separei pedidos entregues dos que não têm data real.
- [ ] Comparei o atraso informado com a diferença entre as datas.
- [ ] Relacionei cada gráfico a uma pergunta analítica.
- [ ] Interpretei valores negativos, zero e positivos corretamente.
- [ ] Comparei grupos considerando medidas e quantidade de observações.
- [ ] Tratei os resultados como descritivos, sem atribuir causalidade.
